# 전처리

In [ ]:
# 데이터 병합

import pandas as pd

# 1. 파일 불러오기
df1 = pd.read_csv(
    "secom_labels.data",
    sep=r"\s+",
    header=None
)

df2 = pd.read_csv(
    "secom.data",
    sep=r"\s+",
    header=None
)

# 2. 열 방향 결합
df = pd.concat([df1, df2], axis=1)

# 3. 컬럼명 지정
df.columns = (
    ["label", "timestamp"]
    + [f"sensor_{i}" for i in range(df2.shape[1])]
)

# 4. 새로운 CSV 파일로 저장
df.to_csv("secom_merged.csv", index=False)

# 5. 확인
print("성공적으로 저장되었습니다!")
print("데이터 크기:", df.shape)
print("\n컬럼명:")
print(df.columns.tolist()[:10])
print("...")
print(df.columns.tolist()[-10:])

### secom_merged.csv 데이터를 확인하여 결측률 50% 초과 센서 제거 및 분산0 센서 제거

In [5]:
import pandas as pd

In [4]:
df = pd.read_csv("secom_merged.csv")

print(df.shape)
print(df.head())

(1567, 592)
   label            timestamp  sensor_0  sensor_1   sensor_2   sensor_3  \
0     -1  19/07/2008 11:55:00   3030.93   2564.00  2187.7333  1411.1265   
1     -1  19/07/2008 12:32:00   3095.78   2465.14  2230.4222  1463.6606   
2      1  19/07/2008 13:17:00   2932.61   2559.94  2186.4111  1698.0172   
3     -1  19/07/2008 14:43:00   2988.72   2479.90  2199.0333   909.7926   
4     -1  19/07/2008 15:22:00   3032.24   2502.87  2233.3667  1326.5200   

   sensor_4  sensor_5  sensor_6  sensor_7  ...  sensor_580  sensor_581  \
0    1.3602     100.0   97.6133    0.1242  ...         NaN         NaN   
1    0.8294     100.0  102.3433    0.1247  ...      0.0060    208.2045   
2    1.5102     100.0   95.4878    0.1241  ...      0.0148     82.8602   
3    1.3204     100.0  104.2367    0.1217  ...      0.0044     73.8432   
4    1.5334     100.0  100.3967    0.1235  ...         NaN         NaN   

   sensor_582  sensor_583  sensor_584  sensor_585  sensor_586  sensor_587  \
0      0.5005  

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, label to sensor_589
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB


In [7]:
#센서 분석

sensor_cols = [col for col in df.columns if col.startswith("sensor_")]

print("센서 개수:", len(sensor_cols))

센서 개수: 590


In [8]:
X = df[sensor_cols]

print(X.shape)

(1567, 590)


In [9]:
# 결측치 확인
print("전체 결측치 개수:", X.isna().sum().sum())

전체 결측치 개수: 41951


In [13]:
missing_ratio.sort_values(ascending=False).head(20)

sensor_292    0.911934
sensor_293    0.911934
sensor_158    0.911934
sensor_157    0.911934
sensor_492    0.855775
sensor_85     0.855775
sensor_358    0.855775
sensor_220    0.855775
sensor_244    0.649649
sensor_517    0.649649
sensor_109    0.649649
sensor_246    0.649649
sensor_383    0.649649
sensor_382    0.649649
sensor_384    0.649649
sensor_245    0.649649
sensor_111    0.649649
sensor_110    0.649649
sensor_516    0.649649
sensor_518    0.649649
dtype: float64

In [16]:
#결측률 50퍼 초과 센서 개수

high_missing = missing_ratio > 0.5

print("결측률 50% 초과 센서:", high_missing.sum())

결측률 50% 초과 센서: 28


In [19]:
# 50% 초과 센서 28개가 나와서 제거

keep_cols = missing_ratio <= 0.5

X_clean = X.loc[:, keep_cols]

print("원래 센서 수:", X.shape[1])
print("제거 후 센서 수:", X_clean.shape[1])
print("제거한 센서 수:", X.shape[1] - X_clean.shape[1])

원래 센서 수: 590
제거 후 센서 수: 562
제거한 센서 수: 28


In [22]:
# 아무런 정보도 없는 분산0 센서 찾기

variance = X_clean.var()

zero_var_cols = variance[variance == 0].index

print("분산 0 센서:", len(zero_var_cols))

분산 0 센서: 116


In [24]:
# 분산0 센서  116개 제거

X_clean = X_clean.drop(columns=zero_var_cols)

print("최종 센서 수:", X_clean.shape[1])

최종 센서 수: 446


In [26]:
# 보기 쉽게 정리

original_sensor_count = len(sensor_cols)

missing_removed = (missing_ratio > 0.5).sum()

zero_variance_removed = len(zero_var_cols)

final_sensor_count = X_clean.shape[1]

summary = pd.DataFrame({
    "단계": [
        "원본",
        "결측률 50% 초과 제거",
        "분산 0 제거"
    ],
    "센서 수": [
        original_sensor_count,
        original_sensor_count - missing_removed,
        final_sensor_count
    ],
    "제거 센서 수": [
        0,
        missing_removed,
        zero_variance_removed
    ]
})

summary

,단계,센서 수,제거 센서 수
0,원본,590,0
1,결측률 50% 초과 제거,562,28
2,분산 0 제거,446,116


In [27]:
# 다시 합치기
df_clean = pd.concat(
    [
        df[["label", "timestamp"]],
        X_clean
    ],
    axis=1
)

In [28]:
print(df_clean.shape)
print(df_clean.head())

(1567, 448)
   label            timestamp  sensor_0  sensor_1   sensor_2   sensor_3  \
0     -1  19/07/2008 11:55:00   3030.93   2564.00  2187.7333  1411.1265   
1     -1  19/07/2008 12:32:00   3095.78   2465.14  2230.4222  1463.6606   
2      1  19/07/2008 13:17:00   2932.61   2559.94  2186.4111  1698.0172   
3     -1  19/07/2008 14:43:00   2988.72   2479.90  2199.0333   909.7926   
4     -1  19/07/2008 15:22:00   3032.24   2502.87  2233.3667  1326.5200   

   sensor_4  sensor_6  sensor_7  sensor_8  ...  sensor_576  sensor_577  \
0    1.3602   97.6133    0.1242    1.5005  ...      1.6765     14.9509   
1    0.8294  102.3433    0.1247    1.4966  ...      1.1065     10.9003   
2    1.5102   95.4878    0.1241    1.4436  ...      2.0952      9.2721   
3    1.3204  104.2367    0.1217    1.4882  ...      1.7585      8.5831   
4    1.5334  100.3967    0.1235    1.5031  ...      1.6597     10.9698   

   sensor_582  sensor_583  sensor_584  sensor_585  sensor_586  sensor_587  \
0      0.5005  

In [29]:
# 최종저장
df_clean.to_csv("secom_cleaned.csv",index=False)

print("\n최종 데이터 저장 완료!")
print("파일명: secom_cleaned.csv")


최종 데이터 저장 완료!
파일명: secom_cleaned.csv
